# Defining Resources

**Resources** expose data to clients, much like **GET handlers** in an HTTP server. Use them when you need to *fetch information* rather than *perform an action* (that's what tools are for).

> Implement these in **`cli-project/mcp_server.py`** (the two resource `TODO`s). `skip-worktree` keeps your edits local; the answer key is in `cli-project-complete/`.

## Motivating example: `@document` mentions

To let users type `@document_name` to reference a file, you need two reads:

1. **List all documents** — for autocomplete.
2. **Fetch one document's contents** — when it's mentioned.

When a user mentions a document, the system **injects the contents directly into the prompt** sent to Claude — so Claude doesn't need a tool call to fetch it. Fetching data = resource; taking an action = tool.

## How resources work

Request-response by **URI**. The client sends a **`ReadResourceRequest`** with a URI naming the resource; the server processes it and returns a **`ReadResourceResult`**.

Flow: your code requests a resource from the MCP client → client forwards to the server → server matches the URI, runs the function, returns the result.

## Direct resources (static URI)

Fixed URI, no parameters — good for parameter-free reads like "list everything":

```python
@mcp.resource(
    "docs://documents",
    mime_type="application/json"
)
def list_docs() -> list[str]:
    return list(docs.keys())
```

## Templated resources (URI parameters)

Parameters embedded in the URI. The SDK parses them and passes them as keyword arguments to your function:

```python
@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain"
)
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]
```

`{doc_id}` in the URI becomes the `doc_id` argument.

## Implementation details

Resources can return any data — strings, JSON, binary. `mime_type` hints the client at the shape:

| `mime_type` | For |
|-------------|-----|
| `application/json` | structured data |
| `text/plain` | plain text |
| `application/pdf` | binary files |

The SDK **auto-serializes** your return value — return the data structure directly; no manual `json.dumps`.

## Testing resources

Start the server and open the inspector:

```bash
uv run mcp dev mcp_server.py
```

Two sections appear:

- **Resources** — your direct/static resources.
- **Resource Templates** — your templated resources.

Click one to test it; for templated resources, supply the parameter values. The inspector shows the exact response your client will receive — MIME type and serialized data.

> Contrast with tools: resources are **read-only GETs** (no side effects), so they're cheaper and don't require Claude to decide on a tool call — the app fetches and injects directly.

Next: **Accessing resources** — call these from the client side (`read_resource`).